In [1]:
# CELL 1 — Environment, fixed seeds, T4 verification, and memory references
import hashlib, json, os, random, sys, time, urllib.request
from collections import defaultdict
from pathlib import Path

SEED = 20260629
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_grad_enabled(False)

assert not torch.is_grad_enabled()
assert torch.cuda.is_available(), "Enable Accelerator: GPU T4 x1 in Kaggle Settings."
assert torch.cuda.device_count() == 1, f"Expected one visible GPU, found {torch.cuda.device_count()}."
GPU_NAME = torch.cuda.get_device_name(0)
assert "T4" in GPU_NAME.upper(), f"This official profile requires one NVIDIA T4; found {GPU_NAME}."
T4_CAPACITY_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30

def measure_adamw_state_allocation(label, parameters):
    """Measured FP16 parameter/gradient plus FP32 first/second moments."""
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    tensors = [torch.empty(parameters, device="cuda", dtype=torch.float16) for _ in range(2)]
    tensors += [torch.empty(parameters, device="cuda", dtype=torch.float32) for _ in range(2)]
    torch.cuda.synchronize()
    row = {"label": label, "parameters": parameters,
           "measured_peak_gib": torch.cuda.max_memory_allocated() / 2**30}
    del tensors; torch.cuda.empty_cache()
    return row

ADAMW_EMPIRICAL_PROFILE = [
    measure_adamw_state_allocation("AdamW 25M state allocation — measured", 25_000_000),
    measure_adamw_state_allocation("AdamW 75M state allocation — measured", 75_000_000),
]
print({"cell": 1, "seed": SEED, "gpu": GPU_NAME, "t4_gib": T4_CAPACITY_GIB,
       "grad_enabled": torch.is_grad_enabled(), "adamw_profiles": ADAMW_EMPIRICAL_PROFILE})



{'cell': 1, 'seed': 20260629, 'gpu': 'Tesla T4', 't4_gib': 14.56219482421875, 'grad_enabled': False, 'adamw_profiles': [{'label': 'AdamW 25M state allocation — measured', 'parameters': 25000000, 'measured_peak_gib': 0.28125}, {'label': 'AdamW 75M state allocation — measured', 'parameters': 75000000, 'measured_peak_gib': 0.8400440216064453}]}


In [2]:
# CELL 2 — Prepared corpus plus frozen WikiText-103 and MT-Bench ingestion
INPUT_ROOT = Path("/kaggle/input")
manifest_candidates = sorted(INPUT_ROOT.rglob("artifact_manifest.json"))
prepared = []
for candidate in manifest_candidates:
    try:
        payload = json.loads(candidate.read_text(encoding="utf-8"))
    except Exception:
        continue
    if payload.get("schema_version") == 4 and payload.get("tokenizer_algorithm") == "utf8_byte_v1":
        prepared.append((candidate, payload))
assert prepared, "Attach the V7 byte prepared-data Kaggle Dataset."
# Prefer a Dataset input over notebook-output duplicates, then the largest corpus.
prepared.sort(key=lambda item: ("/datasets/" in str(item[0]).replace("\\", "/"),
                                int(item[1].get("train_tokens", 0))), reverse=True)
MANIFEST_PATH, manifest = prepared[0]
DATA = MANIFEST_PATH.parent
assert manifest["seed"] == SEED and manifest["vocab"] == 32768
assert manifest["active_token_ids"] == 264
ACTIVE_TOKEN_IDS = int(manifest["active_token_ids"])

for name, info in manifest["files"].items():
    path = DATA / name
    assert path.is_file() and path.stat().st_size == info["bytes"], f"Prepared artifact mismatch: {name}"
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""): digest.update(block)
    assert digest.hexdigest() == info["sha256"], f"SHA-256 mismatch: {name}"

train_data = np.memmap(DATA / "fineweb_edu_train.uint16", dtype=np.uint16, mode="r")
valid_data = np.memmap(DATA / "fineweb_edu_validation.uint16", dtype=np.uint16, mode="r")
with (DATA / "oasst1_pairs.jsonl").open("r", encoding="utf-8") as handle:
    chat_rows = [json.loads(line) for line in handle if line.strip()]
with (DATA / "frozen_benchmarks_tokenized.jsonl").open("r", encoding="utf-8") as handle:
    all_choice_rows = [json.loads(line) for line in handle if line.strip()]
choice_rows = [row for row in all_choice_rows if row["task"] in ("hellaswag", "piqa")]

BENCH_SHA = {
    "mt_bench_question.jsonl": "119565adbab82227089cefdb44c8d7e2cf04dc0a0ec233634c82e7d4e2a944f7",
    "wikitext103_test.parquet": "5f1bea067869d04849c0f975a2b29c4ff47d867f484f5010ea5e861eab246d91",
}
BENCH_URL = {
    "mt_bench_question.jsonl": "https://raw.githubusercontent.com/lm-sys/FastChat/main/fastchat/llm_judge/data/mt_bench/question.jsonl",
    "wikitext103_test.parquet": "https://huggingface.co/datasets/Salesforce/wikitext/resolve/refs%2Fconvert%2Fparquet/wikitext-103-raw-v1/test/0000.parquet",
}
BENCH_CACHE = Path("/kaggle/working/frozen_four_benchmark_inputs")
BENCH_CACHE.mkdir(parents=True, exist_ok=True)

def locate_or_download(name):
    hits = sorted(INPUT_ROOT.rglob(name))
    source = hits[0] if hits else BENCH_CACHE / name
    if not hits and not source.is_file():
        print({"download": name, "url": BENCH_URL[name]}, flush=True)
        urllib.request.urlretrieve(BENCH_URL[name], source)
    assert source.is_file(), f"Missing {name}; attach the frozen benchmark Dataset or enable Internet."
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    assert digest == BENCH_SHA[name], f"Frozen benchmark SHA-256 mismatch: {name}"
    return source

MT_PATH = locate_or_download("mt_bench_question.jsonl")
WIKI_PATH = locate_or_download("wikitext103_test.parquet")
with MT_PATH.open("r", encoding="utf-8") as handle:
    mt_questions = [json.loads(line) for line in handle if line.strip()]
assert len(mt_questions) == 80 and all(len(row["turns"]) == 2 for row in mt_questions)

import pyarrow.parquet as pq
wiki_table = pq.read_table(WIKI_PATH, columns=["text"])
wiki_text = "\n".join(text for text in wiki_table.column("text").to_pylist() if text)

BYTE_OFFSET = 4
SPECIAL = {"<|system|>": 260, "<|user|>": 261, "<|assistant|>": 262, "<|turn_end|>": 263}
def encode_text(text):
    return [byte + BYTE_OFFSET for byte in str(text).encode("utf-8", errors="replace")]
def decode_ids(ids):
    chunks, raw = [], bytearray()
    reverse = {value: key for key, value in SPECIAL.items()}
    for token in ids:
        if 4 <= token < 260: raw.append(token - 4)
        elif token in reverse:
            if raw: chunks.append(bytes(raw).decode("utf-8", errors="replace")); raw.clear()
            chunks.append(reverse[token])
    if raw: chunks.append(bytes(raw).decode("utf-8", errors="replace"))
    return "".join(chunks)

wiki_ids = np.asarray(encode_text(wiki_text), dtype=np.uint16)
assert len(train_data) == manifest["train_tokens"] and chat_rows and choice_rows and len(wiki_ids) > 100_000
CORE_START = time.perf_counter()  # initialization + training + all evaluation + save
print({"cell": 2, "prepared_data": str(DATA), "train_tokens": len(train_data),
       "chat_pairs": len(chat_rows), "choice_records": len(choice_rows),
       "wikitext_test_bytes": len(wiki_ids), "mt_bench_questions": len(mt_questions),
       "duplicate_prepared_manifests_ignored": max(0, len(prepared)-1)})



{'cell': 2, 'prepared_data': '/kaggle/input/datasets/anuveshikaprasad/local-ff-4b-prepared-data-v7-byte/prepared_data', 'train_tokens': 100007268, 'chat_pairs': 24002, 'choice_records': 11880, 'wikitext_test_bytes': 1290546, 'mt_bench_questions': 80, 'duplicate_prepared_manifests_ignored': 0}


In [3]:
# CELL 3 — Audit-visible 4.105B random initialization plus local decoder
MODEL_SOURCE_FOR_AUDIT = '"""Sparse local Forward-Forward MoE with an independent local byte decoder."""\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport json\nimport torch\n\ntorch.set_grad_enabled(False)\n\n\ndef sigmoid(x):\n    return torch.sigmoid(x)\n\n\ndef rmsnorm(x, eps=1e-6):\n    scale = torch.rsqrt(x.float().square().mean(-1, keepdim=True) + eps)\n    return x * scale.to(x.dtype)\n\n\ndef prefix_mean(x):\n    total = x.float().cumsum(1) - x.float()\n    den = torch.arange(x.shape[1], device=x.device, dtype=torch.float32).clamp_min_(1)\n    value = (total / den[None, :, None]).to(x.dtype)\n    value[:, 0].zero_()\n    return value\n\n\ndef plasticity_(weight, local_derivative, rate, clip=4.0):\n    derivative32 = local_derivative.float()\n    normalizer = derivative32.square().mean().sqrt().clamp_min_(1e-6)\n    change32 = derivative32.div(normalizer).clamp_(-clip, clip)\n    change32.nan_to_num_(nan=0.0, posinf=clip, neginf=-clip)\n    weight.add_(change32.to(weight.dtype), alpha=-rate)\n\n\n@dataclass\nclass Config:\n    vocab: int = 512\n    active_tokens: int = 264\n    width: int = 64\n    depth: int = 2\n    experts: int = 4\n    expert_hidden: int = 128\n    threshold: float = 1.0\n    rate: float = 1e-3\n    router_rate: float = 2e-4\n    decoder_rate: float = 2e-4\n    seed: int = 20260629\n\n    @classmethod\n    def nominal_4b(cls, seed=20260629):\n        return cls(vocab=32768, active_tokens=264, width=1024, depth=20,\n                   experts=24, expert_hidden=4096, threshold=1.0,\n                   rate=1e-3, router_rate=2e-4, decoder_rate=2e-4,\n                   seed=seed)\n\n    def backbone_parameter_count(self):\n        emb = self.vocab * self.width\n        per_expert = (self.width * self.expert_hidden +\n                      self.expert_hidden * self.width +\n                      self.expert_hidden + self.width)\n        per_block = (2 * self.width * self.width + self.width +\n                     self.width * self.experts + self.experts +\n                     self.experts * per_expert)\n        return emb + self.depth * per_block\n\n    def parameter_count(self):\n        return self.backbone_parameter_count() + self.width * self.active_tokens + self.active_tokens\n\n\nclass SparseLocalBlock:\n    def __init__(self, cfg, device, dtype, generator):\n        d, h, e = cfg.width, cfg.expert_hidden, cfg.experts\n        self.cfg = cfg\n        self.mix = torch.randn(2*d, d, device=device, dtype=dtype, generator=generator).mul_(d ** -0.5)\n        self.bias = torch.zeros(d, device=device, dtype=dtype)\n        self.router = torch.randn(d, e, device=device, dtype=dtype, generator=generator).mul_(d ** -0.5)\n        self.router_bias = torch.zeros(e, device=device, dtype=torch.float32)\n        self.up = torch.randn(e, d, h, device=device, dtype=dtype, generator=generator).mul_(d ** -0.5)\n        self.down = torch.randn(e, h, d, device=device, dtype=dtype, generator=generator).mul_(h ** -0.5)\n        self.up_bias = torch.zeros(e, h, device=device, dtype=dtype)\n        self.down_bias = torch.zeros(e, d, device=device, dtype=dtype)\n\n    def _stem(self, x):\n        q = torch.cat((x, prefix_mean(x)), -1)\n        pre = torch.matmul(q, self.mix).add_(self.bias)\n        s = sigmoid(pre)\n        stem = pre * s\n        route = (torch.matmul(stem, self.router).float() + self.router_bias).argmax(-1)\n        return q, pre, s, stem, route\n\n    def _expert_side(self, stem_rows, expert_id):\n        pre = torch.matmul(stem_rows, self.up[expert_id]).add_(self.up_bias[expert_id])\n        s = sigmoid(pre)\n        hidden = pre * s\n        out = stem_rows + torch.matmul(hidden, self.down[expert_id]).add_(self.down_bias[expert_id])\n        goodness = out.float().square().mean(-1)\n        return pre, s, hidden, out, goodness\n\n    def learn(self, positive, negative):\n        qp, pp, sp, hp, rp = self._stem(positive)\n        qn, pn, sn, hn, rn = self._stem(negative)\n        shape = hp.shape\n        hp_flat, hn_flat = hp.reshape(-1, shape[-1]), hn.reshape(-1, shape[-1])\n        rp_flat, rn_flat = rp.reshape(-1), rn.reshape(-1)\n        outp, outn = torch.empty_like(hp_flat), torch.empty_like(hn_flat)\n        dhp, dhn = torch.zeros_like(hp_flat), torch.zeros_like(hn_flat)\n        losses, correct, observations = [], 0.0, 0\n        for expert in range(self.cfg.experts):\n            ip = torch.nonzero(rp_flat == expert, as_tuple=False).flatten()\n            inn = torch.nonzero(rn_flat == expert, as_tuple=False).flatten()\n            if ip.numel() == 0 and inn.numel() == 0:\n                continue\n            grad_up = torch.zeros_like(self.up[expert]); grad_down = torch.zeros_like(self.down[expert])\n            grad_ub = torch.zeros_like(self.up_bias[expert]); grad_db = torch.zeros_like(self.down_bias[expert])\n            if ip.numel():\n                ap, sap, vp, op, gp = self._expert_side(hp_flat[ip], expert)\n                alpha = -sigmoid(self.cfg.threshold - gp).div_(gp.numel())\n                dup = (alpha[:, None] * (2.0 / self.cfg.width) * op.float()).to(hp.dtype)\n                dv = torch.matmul(dup, self.down[expert].T)\n                da = dv * sap * (1 + ap * (1 - sap))\n                grad_down.add_(torch.matmul(vp.T, dup)); grad_up.add_(torch.matmul(hp_flat[ip].T, da))\n                grad_db.add_(dup.sum(0)); grad_ub.add_(da.sum(0))\n                dhp[ip] = dup + torch.matmul(da, self.up[expert].T); outp[ip] = op\n                losses.append(torch.nn.functional.softplus(self.cfg.threshold - gp).mean())\n                correct += float((gp > self.cfg.threshold).sum()); observations += gp.numel()\n                reward = sigmoid(self.cfg.threshold - gp).to(hp.dtype)\n                self.router[:, expert].add_(torch.matmul(hp_flat[ip].T, reward[:, None]).squeeze(1) / ip.numel(),\n                                            alpha=self.cfg.router_rate)\n            if inn.numel():\n                an, san, vn, on, gn = self._expert_side(hn_flat[inn], expert)\n                alpha = sigmoid(gn - self.cfg.threshold).div_(gn.numel())\n                dun = (alpha[:, None] * (2.0 / self.cfg.width) * on.float()).to(hn.dtype)\n                dv = torch.matmul(dun, self.down[expert].T)\n                da = dv * san * (1 + an * (1 - san))\n                grad_down.add_(torch.matmul(vn.T, dun)); grad_up.add_(torch.matmul(hn_flat[inn].T, da))\n                grad_db.add_(dun.sum(0)); grad_ub.add_(da.sum(0))\n                dhn[inn] = dun + torch.matmul(da, self.up[expert].T); outn[inn] = on\n                losses.append(torch.nn.functional.softplus(gn - self.cfg.threshold).mean())\n                correct += float((gn < self.cfg.threshold).sum()); observations += gn.numel()\n                reward = -sigmoid(gn - self.cfg.threshold).to(hn.dtype)\n                self.router[:, expert].add_(torch.matmul(hn_flat[inn].T, reward[:, None]).squeeze(1) / inn.numel(),\n                                            alpha=self.cfg.router_rate)\n            plasticity_(self.down[expert], grad_down, self.cfg.rate)\n            plasticity_(self.up[expert], grad_up, self.cfg.rate)\n            plasticity_(self.down_bias[expert], grad_db, self.cfg.rate)\n            plasticity_(self.up_bias[expert], grad_ub, self.cfg.rate)\n        counts = torch.bincount(torch.cat((rp_flat, rn_flat)), minlength=self.cfg.experts).float()\n        self.router_bias.add_((counts.mean() - counts) / counts.sum().clamp_min(1), alpha=self.cfg.router_rate)\n        dhp, dhn = dhp.reshape(shape), dhn.reshape(shape)\n        dpp = dhp * sp * (1 + pp * (1 - sp)); dpn = dhn * sn * (1 + pn * (1 - sn))\n        grad_mix = torch.matmul(qp.reshape(-1, qp.shape[-1]).T, dpp.reshape(-1, dpp.shape[-1]))\n        grad_mix.add_(torch.matmul(qn.reshape(-1, qn.shape[-1]).T, dpn.reshape(-1, dpn.shape[-1])))\n        plasticity_(self.mix, grad_mix, self.cfg.rate)\n        plasticity_(self.bias, dpp.sum((0, 1)) + dpn.sum((0, 1)), self.cfg.rate)\n        mean_loss = torch.stack(losses).mean().item() if losses else 0.0\n        return rmsnorm(outp.reshape(shape)), rmsnorm(outn.reshape(shape)), mean_loss, correct / max(observations, 1)\n\n    def forward(self, x, return_goodness=False):\n        _, _, _, stem, route = self._stem(x)\n        shape = stem.shape; flat, ids = stem.reshape(-1, shape[-1]), route.reshape(-1)\n        out = torch.empty_like(flat)\n        for expert in range(self.cfg.experts):\n            idx = torch.nonzero(ids == expert, as_tuple=False).flatten()\n            if idx.numel():\n                _, _, _, value, _ = self._expert_side(flat[idx], expert); out[idx] = value\n        out = out.reshape(shape)\n        goodness = out.float().square().mean(-1); normalized = rmsnorm(out)\n        return (normalized, goodness) if return_goodness else normalized\n\n    def forward_one(self, x, prefix):\n        q = torch.cat((x, prefix), -1)\n        pre = torch.matmul(q, self.mix).add_(self.bias)\n        stem = pre * sigmoid(pre)\n        expert = int((torch.matmul(stem, self.router).float() + self.router_bias).argmax(-1).item())\n        _, _, _, out, _ = self._expert_side(stem, expert)\n        return rmsnorm(out)\n\n    def tensors(self):\n        return {"mix": self.mix, "bias": self.bias, "router": self.router,\n                "router_bias": self.router_bias, "up": self.up, "down": self.down,\n                "up_bias": self.up_bias, "down_bias": self.down_bias}\n\n\nclass LocalByteDecoder:\n    def __init__(self, cfg, device, dtype, generator):\n        self.cfg = cfg\n        self.weight = torch.randn(cfg.width, cfg.active_tokens, device=device, dtype=dtype,\n                                  generator=generator).mul_(cfg.width ** -0.5)\n        self.bias = torch.zeros(cfg.active_tokens, device=device, dtype=torch.float32)\n\n    def logits(self, hidden):\n        return torch.matmul(hidden.float(), self.weight.float()) + self.bias\n\n    def learn(self, hidden, ids):\n        x = hidden[:, :-1].reshape(-1, self.cfg.width).float()\n        targets = ids[:, 1:].reshape(-1)\n        keep = targets < self.cfg.active_tokens\n        x, targets = x[keep], targets[keep]\n        logits = self.logits(x); logits.sub_(logits.max(-1, keepdim=True).values)\n        exp = logits.exp(); probabilities = exp / exp.sum(-1, keepdim=True).clamp_min_(1e-12)\n        row = torch.arange(targets.numel(), device=targets.device)\n        nll = -torch.log(probabilities[row, targets].clamp_min_(1e-12)).mean()\n        accuracy = (probabilities.argmax(-1) == targets).float().mean()\n        reconstruction_mse = (probabilities.square().sum(-1) -\n                              2 * probabilities[row, targets] + 1).mean() / self.cfg.active_tokens\n        delta = probabilities\n        delta[row, targets] -= 1.0\n        delta.div_(max(targets.numel(), 1))\n        plasticity_(self.weight, torch.matmul(x.T, delta), self.cfg.decoder_rate)\n        plasticity_(self.bias, delta.sum(0), self.cfg.decoder_rate)\n        return float(nll), float(accuracy), float(reconstruction_mse)\n\n    def tensors(self):\n        return {"decoder_weight": self.weight, "decoder_bias": self.bias}\n\n\nclass LocalDecoderMoE4B:\n    def __init__(self, cfg, device):\n        self.cfg, self.device = cfg, device\n        dtype = torch.float16 if device.type == "cuda" else torch.float32\n        gen = torch.Generator(device=device).manual_seed(cfg.seed)\n        self.embedding = rmsnorm(torch.randn(cfg.vocab, cfg.width, device=device, dtype=dtype, generator=gen))\n        self.blocks = [SparseLocalBlock(cfg, device, dtype, gen) for _ in range(cfg.depth)]\n        self.decoder = LocalByteDecoder(cfg, device, dtype, gen)\n        assert all(not tensor.requires_grad for tensor in self.all_tensors())\n\n    def all_tensors(self):\n        yield self.embedding\n        for block in self.blocks: yield from block.tensors().values()\n        yield from self.decoder.tensors().values()\n\n    def learn_pair(self, positive_ids, negative_ids):\n        pos, neg = self.embedding[positive_ids], self.embedding[negative_ids]\n        losses, accuracies = [], []\n        for block in self.blocks:\n            pos, neg, loss, acc = block.learn(pos, neg)\n            losses.append(loss); accuracies.append(acc)\n        decoder_nll, decoder_accuracy, decoder_mse = self.decoder.learn(pos, positive_ids)\n        return (sum(losses)/len(losses), sum(accuracies)/len(accuracies),\n                decoder_nll, decoder_accuracy, decoder_mse)\n\n    def represent(self, ids):\n        x = self.embedding[ids]\n        for block in self.blocks: x = block.forward(x)\n        return x\n\n    def energy(self, ids):\n        x = self.embedding[ids]\n        total = torch.zeros(ids.shape, device=ids.device, dtype=torch.float32)\n        for block in self.blocks:\n            x, goodness = block.forward(x, return_goodness=True); total.add_(goodness)\n        return total / len(self.blocks)\n\n    def token_nll(self, ids):\n        hidden = self.represent(ids)\n        logits = self.decoder.logits(hidden[:, :-1])\n        log_z = torch.logsumexp(logits, -1)\n        target = ids[:, 1:].clamp(0, self.cfg.active_tokens - 1)\n        chosen = logits.gather(-1, target.unsqueeze(-1)).squeeze(-1)\n        return log_z - chosen\n\n    def new_cache(self):\n        return [{"sum": torch.zeros(1, self.cfg.width, device=self.device), "count": 0}\n                for _ in self.blocks]\n\n    def forward_token(self, token_id, cache):\n        x = self.embedding[int(token_id)].reshape(1, -1)\n        for index, block in enumerate(self.blocks):\n            state = cache[index]\n            prefix = (state["sum"] / state["count"]).to(x.dtype) if state["count"] else torch.zeros_like(x)\n            old_x = x\n            x = block.forward_one(x, prefix)\n            state["sum"].add_(old_x.float()); state["count"] += 1\n        return self.decoder.logits(x).squeeze(0)\n\n    def generate(self, prompt_ids, max_new_tokens, generator, temperature=0.8, top_k=40):\n        cache = self.new_cache(); logits = None\n        for token in prompt_ids[-128:]: logits = self.forward_token(token, cache)\n        result = []\n        for _ in range(max_new_tokens):\n            sample_logits = logits.clone(); sample_logits[:3] = -float("inf")\n            values, indices = torch.topk(sample_logits, min(top_k, sample_logits.numel()))\n            probabilities = torch.softmax(values / temperature, 0)\n            token = int(indices[torch.multinomial(probabilities, 1, generator=generator)].item())\n            if token == 3: break\n            result.append(token); logits = self.forward_token(token, cache)\n        return result\n\n    def save_sharded(self, directory, step):\n        directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)\n        (directory / "config.json").write_text(json.dumps(asdict(self.cfg), indent=2), encoding="utf-8")\n        torch.save({"step": step, "embedding": self.embedding.cpu()}, directory / "embedding.pt")\n        self.embedding = self.embedding.to(self.device)\n        for i, block in enumerate(self.blocks):\n            torch.save({k: v.cpu() for k, v in block.tensors().items()}, directory / f"block_{i:02d}.pt")\n        torch.save({k: v.cpu() for k, v in self.decoder.tensors().items()}, directory / "decoder.pt")\n        self.decoder.weight = self.decoder.weight.to(self.device); self.decoder.bias = self.decoder.bias.to(self.device)\n\n    @classmethod\n    def load_sharded(cls, directory, device):\n        directory = Path(directory)\n        cfg = Config(**json.loads((directory / "config.json").read_text(encoding="utf-8")))\n        model = cls(cfg, device)\n        model.embedding = torch.load(directory / "embedding.pt", map_location=device,\n                                     weights_only=True)["embedding"].to(device)\n        for i, block in enumerate(model.blocks):\n            state = torch.load(directory / f"block_{i:02d}.pt", map_location=device, weights_only=True)\n            for key, value in state.items(): setattr(block, key, value.to(device))\n        decoder_state = torch.load(directory / "decoder.pt", map_location=device, weights_only=True)\n        model.decoder.weight = decoder_state["decoder_weight"].to(device)\n        model.decoder.bias = decoder_state["decoder_bias"].to(device)\n        return model\n\n\ndef corrupt(ids, vocab, generator, probability=0.15):\n    mask = torch.rand(ids.shape, device=ids.device, generator=generator) < probability\n    replacement = torch.randint(0, vocab, ids.shape, device=ids.device, generator=generator)\n    result = torch.where(mask, replacement, ids)\n    result[:, -1] = (result[:, -1] + 1) % vocab\n    return result\n'
exec(compile(MODEL_SOURCE_FOR_AUDIT, "embedded_local_decoder_4b.py", "exec"), globals())
device = torch.device("cuda")
cfg = Config.nominal_4b(seed=SEED)
assert cfg.backbone_parameter_count() == 4_104_999_392
assert cfg.parameter_count() == 4_105_269_992
model = LocalDecoderMoE4B(cfg, device)
torch.cuda.synchronize()
print({"cell": 3, "parameters": cfg.parameter_count(),
       "backbone_parameters": cfg.backbone_parameter_count(),
       "local_decoder_parameters": cfg.parameter_count()-cfg.backbone_parameter_count(),
       "allocated_gib": torch.cuda.memory_allocated()/2**30,
       "all_random_or_zero_initialization": True})


{'cell': 3, 'parameters': 4105269992, 'backbone_parameters': 4104999392, 'local_decoder_parameters': 270600, 'allocated_gib': 7.646670818328857, 'all_random_or_zero_initialization': True}


In [4]:
# CELL 4 — Locality audit and mathematical contract
FORBIDDEN = ("backward(", "loss.backward", "torch.autograd", "torch.optim",
             "optim.Adam", "optim.SGD", "jax.grad", "GradientTape")
violations = [marker for marker in FORBIDDEN if marker in MODEL_SOURCE_FOR_AUDIT]
assert not violations, f"Forbidden global-gradient APIs found: {violations}"
assert "def plasticity_" in MODEL_SOURCE_FOR_AUDIT
assert "class LocalByteDecoder" in MODEL_SOURCE_FOR_AUDIT
assert "self.decoder.learn(pos, positive_ids)" in MODEL_SOURCE_FOR_AUDIT
assert "plasticity_(self.weight" in MODEL_SOURCE_FOR_AUDIT
assert all(not tensor.requires_grad for tensor in model.all_tensors())

# Block l uses only its positive/negative activations:
# L_l = mean softplus(theta-g_l(x+)) + mean softplus(g_l(x-)-theta)
# Decoder uses only the final local representation H and observed next bytes:
# D = softmax(H W_d + b_d) - one_hot(next_byte)
# Delta W_d = -eta * normalize(H^T D); D is never sent into any backbone block.
print({"cell": 4, "forbidden_api_violations": violations,
       "all_tensors_require_grad_false": True,
       "backbone_update": "independent layer-local Forward-Forward goodness",
       "decoder_update": "independent local next-byte predictive update",
       "decoder_error_propagated_into_backbone": False})



{'cell': 4, 'forbidden_api_violations': [], 'all_tensors_require_grad_false': True, 'backbone_update': 'independent layer-local Forward-Forward goodness', 'decoder_update': 'independent local next-byte predictive update', 'decoder_error_propagated_into_backbone': False}


In [5]:
# CELL 5 — Wall-clock-bounded pretraining and conversational adaptation
BATCH, LENGTH = 4, 128
TOTAL_SECONDS = 180 * 60
PRETRAIN_END_SECONDS = 90 * 60
CHAT_END_SECONDS = 100 * 60
EVALUATION_END_SECONDS = 165 * 60
CHECKPOINT_RESERVE_SECONDS = 15 * 60
START = CORE_START
HARD_DEADLINE = START + TOTAL_SECONDS
rng = np.random.default_rng(SEED + 1)
negative_generator = torch.Generator(device=device).manual_seed(SEED + 2)
generation_generator = torch.Generator(device=device).manual_seed(SEED + 3)
training_trace, memory_trace = [], []

def sample_windows(data, batch, length, rng, device):
    starts = rng.integers(0, len(data) - length, size=batch)
    array = np.stack([data[s:s+length] for s in starts]).astype(np.int64)
    return torch.from_numpy(array).to(device, non_blocking=True)

def pad_chat(record, length):
    def side(key):
        value = list(record[key][-length:]); return value + [0] * (length-len(value))
    return side("positive"), side("negative")

def capture_probe(model, samples=64):
    snapshots = []
    for tensor in model.all_tensors():
        flat = tensor.reshape(-1); stride = max(flat.numel() // samples, 1)
        snapshots.append(flat[::stride][:samples].clone())
    return snapshots

def update_coverage(model, before, samples=64):
    changed = total = 0
    for tensor, old in zip(model.all_tensors(), before):
        flat = tensor.reshape(-1); stride = max(flat.numel() // samples, 1)
        now = flat[::stride][:samples]
        changed += int((now != old).count_nonzero()); total += now.numel()
    return changed / max(total, 1)

def record_trace(phase, step, tokens, ff_loss, ff_accuracy, decoder_nll, decoder_accuracy, decoder_mse,
                 positive, negative, coverage):
    elapsed = time.perf_counter() - START
    pos_e = model.energy(positive).mean(1); neg_e = model.energy(negative).mean(1)
    margin = pos_e - neg_e
    row = {"phase": phase, "step": step, "tokens": tokens,
           "local_loss": float(ff_loss), "local_accuracy": float(ff_accuracy),
           "decoder_nll": float(decoder_nll), "decoder_accuracy": float(decoder_accuracy),
           "decoder_reconstruction_mse": float(decoder_mse),
           "positive_energy": float(pos_e.mean()), "negative_energy": float(neg_e.mean()),
           "energy_margin": float(margin.mean()),
           "contrastive_accuracy": float((margin > 0).float().mean()),
           "sampled_nonzero_parameter_update_coverage": float(coverage),
           "tokens_per_second": tokens / max(elapsed, 1e-9), "elapsed": elapsed,
           "allocated_gib": torch.cuda.memory_allocated()/2**30,
           "reserved_gib": torch.cuda.memory_reserved()/2**30}
    training_trace.append(row)
    memory_trace.append({k: row[k] for k in ("elapsed", "allocated_gib", "reserved_gib")})
    print(json.dumps(row), flush=True)

tokens = pretrain_steps = chat_steps = 0
while time.perf_counter() < START + PRETRAIN_END_SECONDS:
    positive = sample_windows(train_data, BATCH, LENGTH, rng, device)
    negative = corrupt(positive, ACTIVE_TOKEN_IDS, negative_generator)
    log_now = (pretrain_steps + 1) % 20 == 0
    probe = capture_probe(model) if log_now else None
    ff_loss, ff_accuracy, decoder_nll, decoder_accuracy, decoder_mse = model.learn_pair(positive, negative)
    if not all(np.isfinite(x) for x in (ff_loss, decoder_nll)):
        raise FloatingPointError(f"Non-finite local metric at pretraining step {pretrain_steps+1}")
    tokens += positive.numel(); pretrain_steps += 1
    if log_now:
        record_trace("pretrain", pretrain_steps, tokens, ff_loss, ff_accuracy,
                     decoder_nll, decoder_accuracy, decoder_mse, positive, negative, update_coverage(model, probe))

while time.perf_counter() < START + CHAT_END_SECONDS:
    records = [chat_rows[int(rng.integers(0, len(chat_rows)))] for _ in range(BATCH)]
    pairs = [pad_chat(record, LENGTH) for record in records]
    positive = torch.tensor([pair[0] for pair in pairs], device=device, dtype=torch.long)
    negative = torch.tensor([pair[1] for pair in pairs], device=device, dtype=torch.long)
    log_now = (chat_steps + 1) % 20 == 0
    probe = capture_probe(model) if log_now else None
    ff_loss, ff_accuracy, decoder_nll, decoder_accuracy, decoder_mse = model.learn_pair(positive, negative)
    if not all(np.isfinite(x) for x in (ff_loss, decoder_nll)):
        raise FloatingPointError(f"Non-finite local metric at chat step {chat_steps+1}")
    tokens += positive.numel(); chat_steps += 1
    if log_now:
        record_trace("conversation", chat_steps, tokens, ff_loss, ff_accuracy,
                     decoder_nll, decoder_accuracy, decoder_mse, positive, negative, update_coverage(model, probe))

print({"cell": 5, "pretrain_steps": pretrain_steps, "chat_steps": chat_steps,
       "processed_tokens": tokens, "elapsed_seconds": time.perf_counter()-START})


{"phase": "pretrain", "step": 20, "tokens": 10240, "local_loss": 0.8083905458450318, "local_accuracy": 0.499169921875, "decoder_nll": 5.4732232093811035, "decoder_accuracy": 0.021653544157743454, "decoder_reconstruction_mse": 0.003770830575376749, "positive_energy": 3.2056455612182617, "negative_energy": 3.0894596576690674, "energy_margin": 0.11618602275848389, "contrastive_accuracy": 0.75, "sampled_nonzero_parameter_update_coverage": 0.9327242524916943, "tokens_per_second": 156.28321677485508, "elapsed": 65.52207083600001, "allocated_gib": 7.65469217300415, "reserved_gib": 7.84765625}
{"phase": "pretrain", "step": 40, "tokens": 20480, "local_loss": 1.1912645518779754, "local_accuracy": 0.473193359375, "decoder_nll": 5.88921594619751, "decoder_accuracy": 0.003937007859349251, "decoder_reconstruction_mse": 0.0037910419050604105, "positive_energy": 0.9349325299263, "negative_energy": 0.8511840105056763, "energy_margin": 0.08374848961830139, "contrastive_accuracy": 1.0, "sampled_nonzero_p

In [6]:
# CELL 6 — Native four-benchmark evaluation, figures, and sharded checkpoint
OUTPUT = Path("/kaggle/working/local_decoder_4b_four_benchmarks")
OUTPUT.mkdir(parents=True, exist_ok=True)
EVAL_DEADLINE = START + EVALUATION_END_SECONDS
CHOICE_ROWS_PER_TASK = 256
WIKITEXT_EVAL_BYTES = 32_768
MT_MAX_NEW_BYTES_PER_TURN = 32

def check_eval_deadline(stage):
    if time.perf_counter() >= EVAL_DEADLINE:
        raise RuntimeError(f"Evaluation reserve exhausted during {stage}; run is invalid.")

def local_validation(batches=20):
    hits = total = 0; margins = []
    for _ in range(batches):
        positive = sample_windows(valid_data, BATCH, LENGTH, rng, device)
        negative = corrupt(positive, ACTIVE_TOKEN_IDS, negative_generator)
        pe, ne = model.energy(positive).mean(1), model.energy(negative).mean(1)
        hits += int((pe > ne).sum()); total += len(pe); margins.append(float((pe-ne).mean()))
    return {"n": total, "real_vs_corrupt_accuracy": hits/total,
            "energy_margin": float(np.mean(margins))}

def deterministic_choice_subset(rows, per_task):
    grouped = defaultdict(list)
    for row in rows: grouped[row["task"]].append(row)
    selected = []
    selection_rng = np.random.default_rng(SEED)
    for task in ("hellaswag", "piqa"):
        indices = selection_rng.choice(len(grouped[task]), min(per_task, len(grouped[task])), replace=False)
        selected.extend(grouped[task][int(i)] for i in sorted(indices))
    return selected

def score_multiple_choice(rows):
    outcomes = defaultdict(list)
    for item_index, row in enumerate(rows):
        if item_index % 25 == 0: check_eval_deadline("HellaSwag/PIQA")
        sequences, continuation_lengths = [], []
        for choice in row["choice_ids"]:
            continuation = list(choice) + [3]
            sequence = ([2] + list(row["prompt_ids"]) + continuation)[-LENGTH:]
            sequences.append(sequence); continuation_lengths.append(min(len(continuation), len(sequence)-1))
        width = max(map(len, sequences))
        batch = torch.zeros((len(sequences), width), device=device, dtype=torch.long)
        for i, sequence in enumerate(sequences):
            batch[i, :len(sequence)] = torch.tensor(sequence, device=device)
        nll = model.token_nll(batch)
        scores = []
        for i, sequence in enumerate(sequences):
            count = continuation_lengths[i]
            scores.append(float(nll[i, len(sequence)-count-1:len(sequence)-1].mean()))
        outcomes[row["task"]].append(int(np.argmin(scores) == int(row["label"])))
    return {task: {"n": len(values), "accuracy": float(np.mean(values)),
                   "scoring": "mean local-decoder continuation NLL"}
            for task, values in sorted(outcomes.items())}

def score_wikitext():
    ids = wiki_ids[:min(WIKITEXT_EVAL_BYTES, len(wiki_ids))].astype(np.int64)
    usable = (len(ids)//LENGTH)*LENGTH
    matrix = ids[:usable].reshape(-1, LENGTH)
    total_nll = 0.0; total_targets = 0
    for start in range(0, len(matrix), BATCH):
        if start % (BATCH*16) == 0: check_eval_deadline("WikiText-103")
        batch = torch.from_numpy(matrix[start:start+BATCH].copy()).to(device)
        nll = model.token_nll(batch)
        total_nll += float(nll.sum()); total_targets += nll.numel()
    mean_nll = total_nll / total_targets
    return {"source": "WikiText-103 raw test", "tokenization": "utf8_byte_v1",
            "evaluated_bytes": int(usable), "predicted_bytes": int(total_targets),
            "mean_nll_nats": mean_nll, "byte_perplexity": float(np.exp(min(mean_nll, 50.0))),
            "comparability_warning": "Byte perplexity is not directly comparable to word/subword PPL."}

def mt_prompt(turn1, answer1=None, turn2=None):
    ids = [2, SPECIAL["<|user|>"], *encode_text(turn1), SPECIAL["<|turn_end|>"],
           SPECIAL["<|assistant|>"]]
    if answer1 is not None:
        ids += [*encode_text(answer1), SPECIAL["<|turn_end|>"], SPECIAL["<|user|>"],
                *encode_text(turn2), SPECIAL["<|turn_end|>"], SPECIAL["<|assistant|>"]]
    return ids

def run_mt_bench():
    answers = []
    for index, question in enumerate(mt_questions):
        if index % 5 == 0: check_eval_deadline("MT-Bench generation")
        first_ids = model.generate(mt_prompt(question["turns"][0]), MT_MAX_NEW_BYTES_PER_TURN,
                                   generation_generator)
        first = decode_ids(first_ids).strip()
        second_ids = model.generate(mt_prompt(question["turns"][0], first, question["turns"][1]),
                                    MT_MAX_NEW_BYTES_PER_TURN, generation_generator)
        second = decode_ids(second_ids).strip()
        answers.append({"question_id": question["question_id"],
                        "answer_id": f"local-decoder-{question['question_id']}",
                        "model_id": "local-decoder-4b-scratch",
                        "choices": [{"index": 0, "turns": [first, second]}], "tstamp": 0})
    answer_path = OUTPUT / "mt_bench_answers.jsonl"
    with answer_path.open("w", encoding="utf-8") as handle:
        for answer in answers: handle.write(json.dumps(answer, ensure_ascii=False) + "\n")
    nonempty = sum(bool(turn.strip()) for answer in answers for turn in answer["choices"][0]["turns"])
    return {"questions": len(answers), "turns_generated": len(answers)*2,
            "nonempty_turns": nonempty, "answer_file": str(answer_path),
            "official_judge_score": None,
            "judge_status": "Judge-ready answers generated; no organizer-approved judge supplied."}

validation = local_validation()
choice_benchmark = score_multiple_choice(deterministic_choice_subset(choice_rows, CHOICE_ROWS_PER_TASK))
wikitext = score_wikitext()
mt_bench = run_mt_bench()
check_eval_deadline("post-benchmark checkpoint reserve")

import matplotlib.pyplot as plt
if training_trace:
    minutes = np.asarray([row["elapsed"]/60 for row in training_trace])
    series = lambda key: np.asarray([row[key] for row in training_trace])
    fig, axes = plt.subplots(4, 2, figsize=(16, 18), sharex=True); axes = axes.ravel()
    axes[0].plot(minutes, series("positive_energy"), label="Positive energy")
    axes[0].plot(minutes, series("negative_energy"), label="Negative energy"); axes[0].legend()
    axes[0].set_title("Measured positive and negative energy")
    axes[1].plot(minutes, series("energy_margin"), color="purple"); axes[1].axhline(0, color="black", lw=.8)
    axes[1].set_title("Measured energy margin")
    axes[2].plot(minutes, series("contrastive_accuracy"), label="Contrastive accuracy")
    axes[2].plot(minutes, series("decoder_accuracy"), label="Next-byte accuracy"); axes[2].legend()
    axes[2].set_title("Measured local accuracies")
    axes[3].plot(minutes, series("decoder_reconstruction_mse"), color="brown")
    axes[3].set_title("Decoder one-hot reconstruction MSE")
    axes[4].plot(minutes, series("sampled_nonzero_parameter_update_coverage"), color="green")
    axes[4].set_title("Sampled nonzero update coverage")
    axes[5].plot(minutes, series("tokens_per_second"), color="darkorange")
    axes[5].set_title("Measured effective tokens/second")
    axes[6].plot(minutes, series("allocated_gib"), label="Local decoder 4B allocated — measured")
    axes[6].plot(minutes, series("reserved_gib"), label="Local decoder 4B reserved — measured")
    axes[6].axhline(T4_CAPACITY_GIB, color="black", ls=":", label="T4 capacity — hardware"); axes[6].legend(fontsize=8)
    axes[6].set_title("VRAM over wall-clock time")
    axes[7].plot(minutes, series("local_loss"), label="FF local loss")
    axes[7].plot(minutes, series("decoder_nll"), label="Decoder NLL"); axes[7].legend()
    axes[7].set_title("Measured local objectives")
    for axis in axes: axis.grid(alpha=.25); axis.set_xlabel("Elapsed wall-clock time (minutes)")
    fig.tight_layout(); fig.savefig(OUTPUT/"training_curves.png", dpi=180); plt.close(fig)

peak_allocated = torch.cuda.max_memory_allocated(); peak_reserved = torch.cuda.max_memory_reserved()
adamw_lower_bound = 12 * cfg.parameter_count()
if memory_trace:
    x = [row["elapsed"]/60 for row in memory_trace]
    fig, axis = plt.subplots(figsize=(14, 7))
    axis.plot(x, [row["allocated_gib"] for row in memory_trace], label="Local decoder 4B allocated — measured")
    axis.plot(x, [row["reserved_gib"] for row in memory_trace], label="Local decoder 4B reserved — measured")
    axis.axhline(T4_CAPACITY_GIB, color="black", ls=":", label=f"T4 capacity — hardware ({T4_CAPACITY_GIB:.1f} GiB)")
    axis.axhline(adamw_lower_bound/2**30, color="red", ls="--",
                 label=f"AdamW 12 B/parameter lower bound — analytical ({adamw_lower_bound/2**30:.1f} GiB)")
    for profile in ADAMW_EMPIRICAL_PROFILE:
        axis.axhline(profile["measured_peak_gib"], ls="-.",
                     label=f'{profile["label"]} ({profile["measured_peak_gib"]:.2f} GiB)')
    axis.set(title="Measured local-learning VRAM vs analytical/measured AdamW references",
             xlabel="Elapsed wall-clock time (minutes)", ylabel="GiB")
    axis.grid(alpha=.25); axis.legend(fontsize=8); fig.tight_layout()
    fig.savefig(OUTPUT/"memory_profile.png", dpi=180); plt.close(fig)

if time.perf_counter() > HARD_DEADLINE - CHECKPOINT_RESERVE_SECONDS:
    raise RuntimeError("Insufficient checkpoint reserve before save; run is invalid.")
checkpoint = OUTPUT / "checkpoint"
model.save_sharded(checkpoint, pretrain_steps)
torch.cuda.synchronize()
elapsed = time.perf_counter() - START
if elapsed > TOTAL_SECONDS: raise RuntimeError(f"180-minute limit exceeded: {elapsed:.2f}s")

metrics = {"valid_run": True, "completed_under_180_minutes": True,
           "elapsed_seconds": elapsed, "gpu": GPU_NAME, "seed": SEED,
           "parameters": cfg.parameter_count(), "backbone_parameters": cfg.backbone_parameter_count(),
           "decoder_parameters": cfg.parameter_count()-cfg.backbone_parameter_count(),
           "pretrain_steps": pretrain_steps, "chat_steps": chat_steps, "processed_tokens": tokens,
           "peak_allocated_gib": peak_allocated/2**30, "peak_reserved_gib": peak_reserved/2**30,
           "adamw_12_bytes_per_parameter_lower_bound_gib": adamw_lower_bound/2**30,
           "memory_reduction_vs_adamw_lower_bound": 1-peak_allocated/adamw_lower_bound,
           "validation": validation, "hellaswag": choice_benchmark["hellaswag"],
           "piqa": choice_benchmark["piqa"], "wikitext_103": wikitext, "mt_bench": mt_bench,
           "required_benchmark_status": {"hellaswag": "executed_bounded_subset",
               "piqa": "executed_bounded_subset", "wikitext_103": "executed_byte_perplexity_subset",
               "mt_bench": "all_80_questions_generated_judge_score_not_fabricated"},
           "zero_gradient": True, "decoder_error_propagated_into_backbone": False,
           "random_initialization_only": True, "sota_claim_permitted": False}
(OUTPUT/"final_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(OUTPUT/"training_trace.json").write_text(json.dumps(training_trace), encoding="utf-8")
(OUTPUT/"memory_trace.json").write_text(json.dumps(memory_trace), encoding="utf-8")
print(json.dumps(metrics, indent=2), flush=True)
print({"cell": 6, "checkpoint_blocks": len(list(checkpoint.glob("block_*.pt"))),
       "decoder_checkpoint": (checkpoint/"decoder.pt").is_file(), "output": str(OUTPUT)})



{
  "valid_run": true,
  "completed_under_180_minutes": true,
  "elapsed_seconds": 6343.889441096,
  "gpu": "Tesla T4",
  "seed": 20260629,
  "parameters": 4105269992,
  "backbone_parameters": 4104999392,
  "decoder_parameters": 270600,
  "pretrain_steps": 2848,
  "chat_steps": 322,
  "processed_tokens": 1623040,
  "peak_allocated_gib": 7.760114669799805,
  "peak_reserved_gib": 7.849609375,
  "adamw_12_bytes_per_parameter_lower_bound_gib": 45.87996742129326,
  "memory_reduction_vs_adamw_lower_bound": 0.8308605017404987,
  "validation": {
    "n": 80,
    "real_vs_corrupt_accuracy": 0.525,
    "energy_margin": -0.007811398804187774
  },
  "hellaswag": {
    "n": 256,
    "accuracy": 0.24609375,
    "scoring": "mean local-decoder continuation NLL"
  },
  "piqa": {
    "n": 256,
    "accuracy": 0.51171875,
    "scoring": "mean local-decoder continuation NLL"
  },
  "wikitext_103": {
    "source": "WikiText-103 raw test",
    "tokenization": "utf8_byte_v1",
    "evaluated_bytes": 32768,
  